In [0]:
%sql
create schema retail_capstone.retail;

In [0]:
%sql
create external volume retail_capstone.retail.volume
location 's3://myfirstprojecthkaku/retail/'


Hari

In [0]:
# display the files available to view  
retail_path = '/Volumes/retail_capstone/retail/volume'
display(dbutils.fs.ls(retail_path))

In [0]:
# create source files paths for respective Sales, Customer and sale tran files for reading
s_customer    = f"{retail_path}/customers/"
s_sales       = f"{retail_path}/sales/"
s_products    = f"{retail_path}/products/"


# create taget files paths for respective Sales, Customer and sale tran files for reading
t_customer    = f"{retail_path}/bronze/customers/"
t_sales       = f"{retail_path}/bronze/sales/"
t_products    = f"{retail_path}/bronze/products/"

# create archive files paths for respective Sales, Customer and sale tran files for writing
a_customer    = f"{retail_path}/archive/customers/"
a_sales       = f"{retail_path}/archive/sales/"
a_products    = f"{retail_path}/archive/products/"

In [0]:
sales_df = (spark.read.option("header",True).option("inferSchema",True).csv(s_sales))
display(sales_df)

In [0]:
product_df = (spark.read.option("header",True).option("inferSchema",True).csv(s_products))
display(product_df)

In [0]:
#read the corrrespodning files from S3 Buckets and put in a dataframe
customers_df = spark.read.option("multiline",True).json(s_customer)
display(customers_df)


In [0]:
# create schema for writing each files
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

customer_schema = StructType([
    StructField("city", StringType(), True),
    StructField("country", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("email", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("state", StringType(), True),
    StructField("updated_timestamp", TimestampType(), True)
])
sales_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("order_date", TimestampType(), True),
    StructField("price", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("updated_timestamp", TimestampType(), True)
   ]) 


 
products_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("updated_timestamp", TimestampType(), True)
    ])




In [0]:
#write each of this file to the target bronze location
from pyspark.sql.functions import *
from pyspark.sql.types import *

sales_final_df = spark.createDataFrame([],sales_schema)
for file in dbutils.fs.ls(s_sales):
    sales_df = (spark.read.option("header",True).option("inferSchema",True).csv(file.path))
    sales_df = sales_df.withColumn("updated_timestamp", current_timestamp())
    sales_final_df = sales_final_df.union(sales_df)
    sales_final_df.write.mode("append").parquet(t_sales)

for file in dbutils.fs.ls(s_sales):
     dbutils.fs.mv(file.path, a_sales+file.name)






In [0]:
#write each of the product  file to the target bronze location

products_final_df = spark.createDataFrame([],products_schema)
for file in dbutils.fs.ls(s_products):
    products_df = (spark.read.option("header",True).option("inferSchema",True).csv(file.path))
    products_df = products_df.withColumn("updated_timestamp", current_timestamp())
    products_final_df = products_final_df.union(products_df)
    products_final_df.write.mode("append").parquet(t_products)

for file in dbutils.fs.ls(s_products):
     dbutils.fs.mv(file.path, a_products+file.name)


In [0]:
#write each of the customer  file to the target bronze location

customer_final_df = spark.createDataFrame([],customer_schema)
for file in dbutils.fs.ls(s_customer):
    customer_df = spark.read.option("multiline",True).json(s_customer)
    customer_df = customer_df.withColumn("updated_timestamp", current_timestamp())
    customer_final_df = customer_final_df.union(customer_df)
    customer_final_df.write.mode("append").parquet(t_customer)

for file in dbutils.fs.ls(s_customer):
     dbutils.fs.mv(file.path, a_customer+file.name)